In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib

with open('/content/nutuk.txt') as input_file:
    text_data = input_file.read()

chars = sorted(list(set(text_data)))

torch.set_default_device('cuda')

def encode(str):
    out = []
    for i in str: out.append(chars.index(i))
    return out
def decode(list):
    out = ''
    for i in list: out = f'{out}{chars[i]}'
    return out

block_size = 256
batch_size = 32
embedding_dims = 128
n_head = 8
n_blocks = 6
dropout = 0.2

torch.manual_seed(1234)

split_rate = 0.9
split_idx = int(split_rate * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

val_step = 0

def get_batch(batchsize, blocksize, data):
    idx = torch.randint(len(data) - blocksize, (batchsize, ))
    x = torch.stack([torch.tensor(encode(data[i:i+blocksize])) for i in idx])
    y = torch.stack([torch.tensor(encode(data[i+1:i+blocksize+1])) for i in idx])
    return x,y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dims, head_size, bias=False)
        self.value = nn.Linear(embedding_dims, head_size, bias=False)
        self.query = nn.Linear(embedding_dims, head_size, bias=False)
        self.head_size = head_size
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, ins):
        k = self.key(ins)
        q = self.query(ins)
        wei = q @ k.transpose(-2, -1)
        wei = wei * self.head_size **-0.5
        wei = wei.masked_fill(self.tril[:ins.shape[1], :ins.shape[1]] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(ins)
        out = wei @ v

        return out 


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(embedding_dims, embedding_dims)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4*emb_dim),
            nn.ReLU(),
            nn.Linear(4*emb_dim, emb_dim) # projection
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dims)
        self.positional_encoding = nn.Embedding(block_size, embedding_dims)
        self.blocks = nn.Sequential(
            *[Block(embedding_dims, n_heads=n_head) for _ in range(n_blocks)]
        )
        self.lm_head = nn.Linear(embedding_dims, len(chars))
    def forward(self, idx, targets=None):
        token_embeddings = self.token_embedding_table(idx)
        pos_embeddings = self.positional_encoding(torch.arange(idx.shape[1]))
        x = token_embeddings + pos_embeddings
        x = self.blocks(x)
        
        logits = self.lm_head(x)
        if targets is not None: 
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss
        else:
            return logits, None
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx



model = BigramLanguageModel(len(chars))
batch = get_batch(batch_size, block_size ,train_data)

optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

total_steps = 10000
for step in range(total_steps):
    x, y = get_batch(4,8,train_data)
    logits, loss = model(x,y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    if not step%1000: print(f"{step}/{total_steps}: {loss.item()}")
    optim.step()


xval = torch.tensor([encode(val_data[:-1])])
yval = torch.tensor([encode(val_data[1:])])
loss_val = 0
for _ in range(5000):
    batch = get_batch(batch_size, block_size, val_data)
    _, loss = model(batch[0], batch[1])
    loss_val += loss.item()
    val_step += 1
print(f'val loss: {loss_val/5000}')


ctx = torch.zeros(1,1, dtype=torch.long)
print(decode(model.generate(ctx, 1000)[0].tolist()))

0/10000: 5.325716018676758
1000/10000: 2.484769105911255
2000/10000: 2.681868553161621
3000/10000: 1.9814019203186035
4000/10000: 2.303509473800659
5000/10000: 2.8935012817382812
6000/10000: 2.600489616394043
7000/10000: 2.1568894386291504
8000/10000: 2.269352912902832
9000/10000: 2.101749897003174
val loss: 2.6892621633052824

vil bir ku f cumilansarnanr f datadaftgahakir 
tcutir inagunime inieda binu ku baninan* I hai§u imm r tisa minmiriyokilobenesutafisi- arestistisnantvihayaesu knola mabakayleami mdiselinolanai bar takurastkrvigihu nola- ninLi ininin 
dcacetefiharasrbu ht misedegilan fle hdismefinale 


Padila mi

tare sininase gimedehisirini isafimdaf na; sa9 vai mabitet ticeyanlede- Pa nisestar o fundadavediinadae — nisamahavagini smar. Irde 
steta- 
“Isaa- igini§istetihanayarmhalasi, Ef inlavelaliitmamini- 
ane olalahu" bikadala smiilaz vangimnan cumi§i M 
- vanmtanetazlusia bii- 
sistanis tetavaimdas nca§ecuasisin 
dada$iyi mnin aiza ciili§e 

aredi- §. rahijineyaninisi sgirbu